## Setup

In [8]:
import oci
import json
import base64
import os
import sys
import traceback
import pandas as pd
import oracledb
from dotenv import load_dotenv

load_dotenv('/mnt/c/Git_Repos/oci-ai-playground/.env')
#oracledb.defaults.config_dir = os.environ['TNS_ADMIN']

# PyCharm+WSL: redirect tracebacks to stdout so they appear in cell output
def _show_tb(shell, etype, evalue, tb, tb_offset=None):
    traceback.print_exception(etype, evalue, tb, file=sys.stdout)
    sys.stdout.flush()
get_ipython().set_custom_exc((Exception,), _show_tb)

DATA_DIR = '/mnt/c/Git_Repos/oci-ai-playground/olist_dataset'
print('Setup complete')

Setup complete


## Connect using OML_USER credentials from OCI Vault

In [9]:
config = oci.config.from_file()
secrets_client = oci.secrets.SecretsClient(config)

secret_bundle = secrets_client.get_secret_bundle(os.environ['OML_USER_CREDS_SECRET_OCID'])
creds = json.loads(
    base64.b64decode(secret_bundle.data.secret_bundle_content.content).decode('utf-8')
)

conn = oracledb.connect(
    user=creds['user_name'],
    password=creds['password'],
    dsn=creds['dsn']
)
print(f"Connected as: {creds['user_name']}")
print(f"DB version:   {conn.version}")

Connected as: OML_USER
DB version:   23.26.1.2.0


## Create tables

In [10]:
DDL = {
    'customers': '''
        CREATE TABLE customers (
            customer_id               VARCHAR2(50) PRIMARY KEY,
            customer_unique_id        VARCHAR2(50),
            customer_zip_code_prefix  VARCHAR2(10),
            customer_city             VARCHAR2(100),
            customer_state            VARCHAR2(5)
        )
    ''',
    'geolocation': '''
        CREATE TABLE geolocation (
            geolocation_zip_code_prefix  VARCHAR2(10),
            geolocation_lat              NUMBER(10,6),
            geolocation_lng              NUMBER(10,6),
            geolocation_city             VARCHAR2(100),
            geolocation_state            VARCHAR2(5)
        )
    ''',
    'orders': '''
        CREATE TABLE orders (
            order_id                        VARCHAR2(50) PRIMARY KEY,
            customer_id                     VARCHAR2(50),
            order_status                    VARCHAR2(20),
            order_purchase_timestamp        TIMESTAMP,
            order_approved_at               TIMESTAMP,
            order_delivered_carrier_date    TIMESTAMP,
            order_delivered_customer_date   TIMESTAMP,
            order_estimated_delivery_date   TIMESTAMP
        )
    ''',
    'order_items': '''
        CREATE TABLE order_items (
            order_id             VARCHAR2(50),
            order_item_id        NUMBER,
            product_id           VARCHAR2(50),
            seller_id            VARCHAR2(50),
            shipping_limit_date  TIMESTAMP,
            price                NUMBER(10,2),
            freight_value        NUMBER(10,2)
        )
    ''',
    'order_payments': '''
        CREATE TABLE order_payments (
            order_id              VARCHAR2(50),
            payment_sequential    NUMBER,
            payment_type          VARCHAR2(30),
            payment_installments  NUMBER,
            payment_value         NUMBER(10,2)
        )
    ''',
    'order_reviews': '''
        CREATE TABLE order_reviews (
            review_id               VARCHAR2(50),
            order_id                VARCHAR2(50),
            review_score            NUMBER,
            review_comment_title    VARCHAR2(500),
            review_comment_message  CLOB,
            review_creation_date    TIMESTAMP,
            review_answer_timestamp TIMESTAMP
        )
    ''',
    'products': '''
        CREATE TABLE products (
            product_id                   VARCHAR2(50) PRIMARY KEY,
            product_category_name        VARCHAR2(100),
            product_name_lenght          NUMBER,
            product_description_lenght   NUMBER,
            product_photos_qty           NUMBER,
            product_weight_g             NUMBER,
            product_length_cm            NUMBER,
            product_height_cm            NUMBER,
            product_width_cm             NUMBER
        )
    ''',
    'sellers': '''
        CREATE TABLE sellers (
            seller_id               VARCHAR2(50) PRIMARY KEY,
            seller_zip_code_prefix  VARCHAR2(10),
            seller_city             VARCHAR2(100),
            seller_state            VARCHAR2(5)
        )
    ''',
    'product_category_translation': '''
        CREATE TABLE product_category_translation (
            product_category_name          VARCHAR2(100) PRIMARY KEY,
            product_category_name_english  VARCHAR2(100)
        )
    '''
}

with conn.cursor() as cur:
    for table, ddl in DDL.items():
        # Drop if exists (Oracle has no DROP TABLE IF EXISTS before 23c)
        try:
            cur.execute(f'DROP TABLE {table} PURGE')
            print(f'  Dropped {table}')
        except oracledb.DatabaseError:
            pass  # table didn't exist
        cur.execute(ddl)
        print(f'  Created {table}')

conn.commit()
print('All tables created.')

  Dropped customers
  Created customers
  Dropped geolocation
  Created geolocation
  Dropped orders
  Created orders
  Dropped order_items
  Created order_items
  Dropped order_payments
  Created order_payments
  Dropped order_reviews
  Created order_reviews
  Dropped products
  Created products
  Dropped sellers
  Created sellers
  Dropped product_category_translation
  Created product_category_translation
All tables created.


## Helper: load a DataFrame into a table

In [11]:
def parse_ts(series):
    """Convert a pandas datetime series to Python datetimes, NaT -> None."""
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.apply(lambda x: x.to_pydatetime() if pd.notna(x) else None)

def load_table(conn, table_name, df, insert_sql, batch_size=2000):
    # astype(object) converts numpy scalars to Python native types first,
    # so NaN/NaT -> None replacement is reliable across all dtypes
    df = df.astype(object).where(pd.notnull(df), other=None)
    data = [tuple(row) for row in df.itertuples(index=False, name=None)]
    with conn.cursor() as cur:
        for i in range(0, len(data), batch_size):
            cur.executemany(insert_sql, data[i:i+batch_size])
    conn.commit()
    print(f'  {table_name}: {len(data):,} rows loaded')

## Load datasets

In [12]:
# CUSTOMERS
df = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv', dtype=str)
load_table(conn, 'customers', df,
    'INSERT INTO customers VALUES (:1,:2,:3,:4,:5)')

  customers: 99,441 rows loaded


In [13]:
# SELLERS
df = pd.read_csv(f'{DATA_DIR}/olist_sellers_dataset.csv', dtype=str)
load_table(conn, 'sellers', df,
    'INSERT INTO sellers VALUES (:1,:2,:3,:4)')

  sellers: 3,095 rows loaded


In [14]:
# PRODUCTS
df = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')
num_cols = ['product_name_lenght','product_description_lenght','product_photos_qty',
            'product_weight_g','product_length_cm','product_height_cm','product_width_cm']
df['product_id'] = df['product_id'].astype(str)
df['product_category_name'] = df['product_category_name'].astype(str).where(df['product_category_name'].notna(), other=None)
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
load_table(conn, 'products', df,
    'INSERT INTO products VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9)')

  products: 32,951 rows loaded


In [15]:
# ORDERS
ts_cols = ['order_purchase_timestamp','order_approved_at',
           'order_delivered_carrier_date','order_delivered_customer_date',
           'order_estimated_delivery_date']
df = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv', dtype=str)
for col in ts_cols:
    df[col] = parse_ts(df[col])
load_table(conn, 'orders', df,
    'INSERT INTO orders VALUES (:1,:2,:3,:4,:5,:6,:7,:8)')

  orders: 99,441 rows loaded


In [16]:
# ORDER_ITEMS
df = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
df['order_id'] = df['order_id'].astype(str)
df['product_id'] = df['product_id'].astype(str)
df['seller_id'] = df['seller_id'].astype(str)
df['shipping_limit_date'] = parse_ts(df['shipping_limit_date'])
load_table(conn, 'order_items', df,
    'INSERT INTO order_items VALUES (:1,:2,:3,:4,:5,:6,:7)')

  order_items: 112,650 rows loaded


In [17]:
# ORDER_PAYMENTS
df = pd.read_csv(f'{DATA_DIR}/olist_order_payments_dataset.csv')
df['order_id'] = df['order_id'].astype(str)
df['payment_type'] = df['payment_type'].astype(str)
load_table(conn, 'order_payments', df,
    'INSERT INTO order_payments VALUES (:1,:2,:3,:4,:5)')

  order_payments: 103,886 rows loaded


In [18]:
# ORDER_REVIEWS
df = pd.read_csv(f'{DATA_DIR}/olist_order_reviews_dataset.csv', dtype=str)
df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce')
df['review_creation_date'] = parse_ts(df['review_creation_date'])
df['review_answer_timestamp'] = parse_ts(df['review_answer_timestamp'])
load_table(conn, 'order_reviews', df,
    'INSERT INTO order_reviews VALUES (:1,:2,:3,:4,:5,:6,:7)')

  order_reviews: 99,224 rows loaded


In [19]:
# GEOLOCATION (~1M rows — takes ~30s)
df = pd.read_csv(f'{DATA_DIR}/olist_geolocation_dataset.csv')
df['geolocation_zip_code_prefix'] = df['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
df['geolocation_city'] = df['geolocation_city'].astype(str)
df['geolocation_state'] = df['geolocation_state'].astype(str)
load_table(conn, 'geolocation', df,
    'INSERT INTO geolocation VALUES (:1,:2,:3,:4,:5)', batch_size=5000)

  geolocation: 1,000,163 rows loaded


In [20]:
# PRODUCT_CATEGORY_TRANSLATION
df = pd.read_csv(f'{DATA_DIR}/product_category_name_translation.csv', dtype=str)
df.columns = ['product_category_name', 'product_category_name_english']
load_table(conn, 'product_category_translation', df,
    'INSERT INTO product_category_translation VALUES (:1,:2)')

  product_category_translation: 71 rows loaded


## Verify row counts

In [21]:
tables = ['customers','sellers','products','orders','order_items',
          'order_payments','order_reviews','geolocation','product_category_translation']

with conn.cursor() as cur:
    for t in tables:
        cur.execute(f'SELECT COUNT(*) FROM {t}')
        count = cur.fetchone()[0]
        print(f'  {t:<35} {count:>10,} rows')

  customers                               99,441 rows
  sellers                                  3,095 rows
  products                                32,951 rows
  orders                                  99,441 rows
  order_items                            112,650 rows
  order_payments                         103,886 rows
  order_reviews                           99,224 rows
  geolocation                          1,000,163 rows
  product_category_translation                71 rows
